In [0]:
# ============================================================
# STEP 1 — SETUP
# ============================================================
# Hint:
# We create:
#   1. A schema
#   2. A Volume to store the streaming checkpoint
#   3. The Delta table name
#   4. The checkpoint location
#
# We are NOT creating the Delta table manually.
# writeStream.toTable() will create it for us.
# ============================================================

catalog = "retail"
schema = "sales"

# Create schema if it doesn't already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Create a Volume for the checkpoint
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.streaming_checkpoints
""")

table_name = f"{catalog}.{schema}.device_events"

checkpoint_path = (
    f"/Volumes/{catalog}/{schema}/streaming_checkpoints/device_events"
)

print("Table:     ", table_name)
print("Checkpoint:", checkpoint_path)

Table:      retail.sales.device_events
Checkpoint: /Volumes/retail/sales/streaming_checkpoints/device_events


In [0]:
# ============================================================
# STEP 2 — CLEAN SLATE
# ============================================================
# Hint:
# We stop any previous streaming query from this demo,
# remove the old Delta table, and remove its checkpoint.
#
# Why?
# A streaming checkpoint remembers where the previous query
# was and its state. For a clean learning demo, we start fresh.
# ============================================================

# Stop any active streaming queries
for q in spark.streams.active:
    if q.name == "device_events_ingest":
        q.stop()

# Remove the previous Delta table, if it exists
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Remove the previous checkpoint
dbutils.fs.rm(checkpoint_path, True)

print("Ready for a fresh run.")

Ready for a fresh run.


In [0]:
# ============================================================
# STEP 3 — STREAMING SOURCE
# ============================================================
# Hint:
# The rate source continuously generates rows.
#
# rowsPerSecond = 5
# means Spark will generate approximately 5 new rows per second.
#
# value is a continuously increasing number:
# 0, 1, 2, 3, 4, 5, ...
#
# We use value % 5 to create 5 device IDs:
# 0, 1, 2, 3, 4
#
# current_timestamp() records when Spark processes the row.
# ============================================================

from pyspark.sql.functions import col, current_timestamp

events_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn(
        "device_id",
        (col("value") % 5).cast("int")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)

# Check whether this is a streaming DataFrame
print("Is this a streaming DataFrame?", events_df.isStreaming)

# Look at the schema
events_df.printSchema()

Is this a streaming DataFrame? True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = false)



In [0]:
# ============================================================
# STEP 4 — WRITE STREAM → DELTA
# ============================================================
# Hint:
# format("delta")
#     → Delta Lake is our streaming sink.
#
# outputMode("append")
#     → Every newly generated event is added as a new row.
#
# checkpointLocation
#     → Spark remembers streaming progress here.
#
# trigger(processingTime="5 seconds")
#     → Spark checks/processes new data every 5 seconds.
#
# toTable(table_name)
#     → Creates/registers the Delta table and writes into it.
#
# IMPORTANT:
# Keep this cell RUNNING.
# Do not stop it yet.
# ============================================================

query = (
    events_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .queryName("device_events_ingest")
    .trigger(processingTime="5 seconds")
    .toTable(table_name)
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5642335438227816>, line 32
      1 # ============================================================
      2 # STEP 4 — WRITE STREAM → DELTA
      3 # ============================================================
   (...)     22 # Do not stop it yet.
     23 # ============================================================
     25 query = (
     26     events_df.writeStream
     27     .format("delta")
     28     .outputMode("append")
     29     .option("checkpointLocation", checkpoint_path)
     30     .queryName("device_events_ingest")
     31     .trigger(processingTime="5 seconds")
---> 32     .toTable(table_name)
     33 )

File <command-5642335438227815>, line 22
      1 # ============================================================
      2 # STEP 3 — STREAMING SOURCE
      3 # ============================================

In [0]:
# ============================================================
# STEP 4A — STOP THE FAILED QUERY
# ============================================================
# Hint:
# The previous query could not start because
# processingTime is not supported on this cluster.
# We stop it before starting the Serverless-compatible version.
# ============================================================

for q in spark.streams.active:
    if q.name == "device_events_ingest":
        q.stop()

print("Ready.")

Ready.


In [0]:
# ============================================================
# STEP 4B — WRITE STREAM → DELTA
# SERVERLESS VERSION
# ============================================================
# Hint:
# availableNow=True means:
#
#   "Process the data currently available,
#    then stop the streaming query."
#
# This is different from processingTime:
#
# processingTime → keeps running continuously
# availableNow   → processes available data and stops
#
# We are using the same Delta sink and checkpoint.
# ============================================================

query = (
    events_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .queryName("device_events_ingest")
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()

print("Stream finished processing available data.")

Stream finished processing available data.


In [0]:
# ============================================================
# STEP 5 — QUERY THE DELTA TABLE
# ============================================================
# Hint:
# The streaming query has written generated events into Delta.
# Now we are reading that Delta table normally with SQL.
# ============================================================

SELECT COUNT(*) AS row_count
FROM retail.sales.device_events;

  File <command-5642335438227820>, line 9
    SELECT COUNT(*) AS row_count
                  ^
SyntaxError: Invalid star expression


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM retail.sales.device_events;

row_count
3


In [0]:
%sql

-- See how many events each device received

SELECT
    device_id,
    COUNT(*) AS events,
    MAX(ingested_at) AS latest
FROM retail.sales.device_events
GROUP BY device_id
ORDER BY device_id;

device_id,events,latest
0,1,2026-10-03T07:00:31.592Z
1,1,2026-10-03T07:00:31.592Z
2,1,2026-10-03T07:00:31.592Z


In [0]:
# ============================================================
# RUN THE STREAM AGAIN
# ============================================================
# Hint:
# availableNow processes the currently available rate-source
# data and then stops.
#
# New data should be added to the existing Delta table.
# ============================================================

query = (
    events_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .queryName("device_events_ingest")
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()

print("Second streaming run finished.")

Second streaming run finished.


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM retail.sales.device_events;

row_count
208


In [0]:
%sql
-- See how many events each device received

SELECT
    device_id,
    COUNT(*) AS events,
    MAX(ingested_at) AS latest
FROM retail.sales.device_events
GROUP BY device_id
ORDER BY device_id;

device_id,events,latest
0,42,2026-10-03T07:01:12.657Z
1,42,2026-10-03T07:01:12.657Z
2,42,2026-10-03T07:01:12.657Z
3,41,2026-10-03T07:01:12.657Z
4,41,2026-10-03T07:01:12.657Z


In [0]:
%sql
sELECT
*
FROM retail.sales.device_events order by value asc;

timestamp,value,device_id,ingested_at
2026-10-03T07:00:31.001Z,0,0,2026-10-03T07:00:31.592Z
2026-10-03T07:00:31.201Z,1,1,2026-10-03T07:00:31.592Z
2026-10-03T07:00:31.401Z,2,2,2026-10-03T07:00:31.592Z
2026-10-03T07:00:31.601Z,3,3,2026-10-03T07:01:12.657Z
2026-10-03T07:00:31.801Z,4,4,2026-10-03T07:01:12.657Z
2026-10-03T07:00:32.001Z,5,0,2026-10-03T07:01:12.657Z
2026-10-03T07:00:32.201Z,6,1,2026-10-03T07:01:12.657Z
2026-10-03T07:00:32.401Z,7,2,2026-10-03T07:01:12.657Z
2026-10-03T07:00:32.601Z,8,3,2026-10-03T07:01:12.657Z
2026-10-03T07:00:32.801Z,9,4,2026-10-03T07:01:12.657Z
